# Notebook 2 — Feature Engineering

Compute technical indicators and inspect the final feature matrix used for LSTM input.

In [1]:
import sys
sys.path.insert(0, '..')

from src.data.fetcher import DataFetcher
from src.data.processor import DataProcessor
from src.utils.config import load_config
import plotly.graph_objects as go
from plotly.subplots import make_subplots

config = load_config()
config['data']['source'] = 'yfinance'
config['data']['timeframe'] = '1Day'

fetcher = DataFetcher(config)
processor = DataProcessor(config)

In [2]:
raw_df = fetcher.fetch('AAPL', start='2020-01-01', end='2024-12-31')
feat_df = processor.process(raw_df)

print(f'Feature columns ({feat_df.shape[1]}):')  
print(feat_df.columns.tolist())
feat_df.head()

[2026-02-25 00:54:36] INFO src.data.fetcher — Loading cached data for AAPL from C:\Users\Arnold\Desktop\Projects\StockPrediction\data\raw\AAPL_1Day_2020-01-01_2024-12-31.parquet
[2026-02-25 00:54:36] INFO src.data.processor — Feature matrix shape after engineering: (1208, 18)
Feature columns (18):
['open', 'high', 'low', 'close', 'volume', 'rsi', 'macd', 'macd_signal', 'macd_diff', 'bb_upper', 'bb_lower', 'bb_width', 'sma_10', 'sma_20', 'sma_50', 'ema_9', 'ema_21', 'volume_change']


,open,high,low,close,volume,rsi,macd,macd_signal,macd_diff,bb_upper,bb_lower,bb_width,sma_10,sma_20,sma_50,ema_9,ema_21,volume_change
datetime,,,,,,,,,,,,,,,,,,
2020-03-13 00:00:00-04:00,64.004286,67.635923,61.119268,67.164749,370732000,45.241592,-2.645107,-2.024405,-0.620702,80.415528,61.000148,27.458597,68.282517,70.707838,73.801324,67.160755,69.941038,-0.114086
2020-03-16 00:00:00-04:00,58.461394,62.600444,57.990225,58.524220,322423600,36.847050,-3.205885,-2.260701,-0.945184,80.087827,59.328637,29.780112,66.914916,69.708232,73.523798,65.433448,68.903146,-0.130305
2020-03-17 00:00:00-04:00,59.804834,62.245254,57.603622,61.097534,324056000,40.394241,-3.403428,-2.489246,-0.914181,79.356482,58.461870,30.321960,66.033950,68.909176,73.311816,64.566265,68.193545,0.005063
2020-03-18 00:00:00-04:00,57.934639,60.406471,57.294328,59.601856,300233600,39.022306,-3.638726,-2.719142,-0.919584,78.263931,57.695107,30.257383,64.679152,67.979519,73.058494,63.573383,67.412482,-0.073513
2020-03-19 00:00:00-04:00,59.775835,61.092695,58.620863,59.145191,271857200,38.591334,-3.818038,-2.938922,-0.879117,77.083455,57.050825,29.869516,63.515965,67.067140,72.802837,62.687745,66.660910,-0.094514


In [3]:
# Visualize RSI and MACD alongside price
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['Close Price', 'RSI', 'MACD'],
                    vertical_spacing=0.05, row_heights=[0.5, 0.25, 0.25])

fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['close'], name='Close'), row=1, col=1)
fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['bb_upper'], name='BB Upper',
                         line=dict(dash='dash', color='gray')), row=1, col=1)
fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['bb_lower'], name='BB Lower',
                         line=dict(dash='dash', color='gray'), fill='tonexty',
                         fillcolor='rgba(200,200,200,0.15)'), row=1, col=1)

fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['rsi'], name='RSI', line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['macd'], name='MACD', line=dict(color='blue')), row=3, col=1)
fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['macd_signal'], name='Signal', line=dict(color='orange')), row=3, col=1)
fig.add_trace(go.Bar(x=feat_df.index, y=feat_df['macd_diff'], name='Histogram'), row=3, col=1)

fig.update_layout(title='AAPL — Technical Indicators', height=800, xaxis_rangeslider_visible=False)
fig.show()

In [4]:
# Correlation heatmap of features
import plotly.figure_factory as ff
import numpy as np

corr = feat_df.corr().round(2)
fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index),
    colorscale='RdBu',
    reversescale=True
)
fig.update_layout(title='Feature Correlation Matrix', height=700)
fig.show()

In [5]:
# Build sequences and inspect shapes
X, y = processor.build_sequences(feat_df, target_col='close')
X_train, y_train, X_val, y_val, X_test, y_test = processor.train_val_test_split(X, y)

print(f'X shape:       {X.shape}  → (samples, seq_len={X.shape[1]}, features={X.shape[2]})')
print(f'X_train:       {X_train.shape}')
print(f'X_val:         {X_val.shape}')
print(f'X_test:        {X_test.shape}')

[2026-02-25 00:55:05] INFO src.data.processor — Split — train: 803, val: 172, test: 173
X shape:       (1148, 60, 18)  → (samples, seq_len=60, features=18)
X_train:       (803, 60, 18)
X_val:         (172, 60, 18)
X_test:        (173, 60, 18)
